# 19 — Fair value gaps

**What to do here:** edit [`nb19_config.py`](nb19_config.py), run all, look at the chart.
Nothing else in this notebook needs touching.

---

A **fair value gap** is a band of price one side skipped. Three candles: if the third's **low** is
still above the first's **high**, nothing traded in between — the middle candle displaced through
it. That band is a *bullish* gap; mirror it for a *bearish* one.

```
bullish:  low[i]  > high[i-2]      band = (high[i-2], low[i])     sits BELOW price
bearish:  high[i] < low[i-2]       band = (high[i],  low[i-2])    sits ABOVE price
```

The gap is knowable at the **close of the third candle** and is drawn from there — never from the
middle candle where it visually sits, which would be a two-bar lookahead painted on the chart.

The notebook draws gaps from several timeframes at once and runs notebook 18's trendline sim twice,
with and without the **FVG veto**, so you can see exactly which trades the veto removed.

> **The veto has been measured and it does not work.**
> [`experiments/fvg_veto_search.py`](../experiments/fvg_veto_search.py): 24 settings × 6 windows,
> scored by R per trade with the top winner trimmed — **0** settings clear all six windows against
> a chance baseline of 0.38, and 4 clear ≥4 where chance predicts 8.2. Below chance.
> See [`docs/WHAT_WORKS.md`](../docs/WHAT_WORKS.md). Treat the comparison below as a description of
> one window, never as a result.

## 1 · Settings

All of them live in [`nb19_config.py`](nb19_config.py) — a plain file next to this notebook, not a
cell, because this notebook is regenerated from a script and a script that states values re-imposes
them on every rebuild.

Edited it while the kernel was running? **Re-run this cell** — it reloads.

In [1]:
import sys, os, importlib, warnings; warnings.filterwarnings('ignore')
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..')); import quant
import numpy as np, pandas as pd

# A kernel that imported `quant` before a feature existed keeps the OLD module in memory while
# Jupyter shows you the NEW file, so the traceback points at code that is not running.
import quant.research as _r
assert hasattr(_r, "fvg_lab"), "Kernel -> Restart: quant was imported before quant.research"

for _p in (os.getcwd(), os.path.join(os.getcwd(), "notebooks"), os.path.abspath("..")):
    if os.path.isfile(os.path.join(_p, "nb19_config.py")) and _p not in sys.path:
        sys.path.insert(0, _p)
import nb19_config as CFG
importlib.reload(CFG)                       # so an edit lands without restarting the kernel

print(f"{CFG.SYMBOL} {CFG.TF}  {CFG.VIEW_START} -> {CFG.END}")
print(f"gaps from {CFG.TF}+{'+'.join(CFG.FVG_TFS)}   mitigation={CFG.FVG_MITIGATION}   "
      f"min size={CFG.FVG_MIN_SIZE_ATR} ATR")
print(f"veto: {CFG.FVG_VETO_MODE if CFG.USE_FVG_VETO else 'off'}")
print(f"settings from {CFG.__file__}")

BTCUSDT 5m  2026-06-01 -> 2026-06-10
gaps from 5m+15m+1h+4h   mitigation=ce   min size=0.5 ATR
veto: off
settings from C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks\nb19_config.py


## 2 · Run

One call. It loads the data, detects the gaps on every timeframe, and runs the sim **twice** — with
the veto and without — so the difference between them is the veto and nothing else.

Everything it builds is on `lab`: `lab.trades` · `lab.all` · `lab.blocked` · `lab.gaps` · `lab.D`.

In [2]:
from quant.research import fvg_lab

lab = fvg_lab(CFG)
lab.summary()

[03:49:48] INFO | quant.data | cache HIT  binance BTCUSDT 5m -> pushdown load
[03:49:49] INFO | quant.data | cache HIT  binance BTCUSDT 1h -> pushdown load
[03:49:49] INFO | quant.data | cache HIT  binance BTCUSDT 15m -> pushdown load
[03:49:49] INFO | quant.data | cache HIT  binance BTCUSDT 4h -> pushdown load
[03:49:49] INFO | quant.data | cache HIT  binance BTCUSDT 1h -> pushdown load
[03:49:49] INFO | quant.data | cache HIT  binance BTCUSDT 15m -> pushdown load
[03:49:49] INFO | quant.data | cache HIT  binance BTCUSDT 1m -> pushdown load
[03:49:51] INFO | quant.engine | backtest done | bars=11,521 trades=3 final_cash=9670.60 return=-3.29% dd=3.29% | 60.5 ms
[03:49:51] INFO | quant.engine | backtest done | bars=11,521 trades=5 final_cash=11015.55 return=10.16% dd=1.72% | 1.2 ms


 tf  gaps  open  median_size_atr  median_bars_to_fill
 5m  1128    62             0.78                  5.0
 4h    52     8             0.77                  6.0
 1h    55     6             0.82                  4.0
15m   108     7             0.80                  6.0

8 trades   +7.0R   50% win    (veto off)


### Which trades the veto removed

The only question worth asking of a filter. Compare `R/trade`, not total R — a veto can only
*remove* trades, so removing them at random lowers total R while leaving the per-trade number alone.

In [3]:
lab.blocked[["tid", "entry_time", "side", "cause", "R", "close_reason"]]

,tid,entry_time,side,cause,R,close_reason


## 3 · Chart

**Gap boxes are drawn behind everything else.** Green is bullish (sits below price), red is bearish
(sits above). A box runs from the bar the gap became knowable to the bar that mitigated it; an
unmitigated gap runs to the right edge.

Three ways to control what you see, in order of how often you will want them:

| control | what it does |
|---|---|
| **`FVGs:` button row** | isolate **one timeframe's** gaps, both directions, in one click |
| **`S/R:` button row** | the same, for support/resistance levels |
| **`EMAs on / off`** | every EMA at once; the legend still picks individuals |
| legend rows | one layer — `4h · FVG bullish`, `1h · S/R zones` … |
| the three dropdowns | timeframe → family → **one individual** gap, zone or line, with its own hover |

Add or remove whole timeframes with `FVG_TFS` and `SR_TFS` in the config — including timeframes
*finer* than the chart, so `("1m", "15m", "1h", "4h")` on a 5m chart works for both.

**S/R zones** are clustered per timeframe: a level fitted on 1h pivots is a different level from one
fitted on 5m pivots, so each frame is clustered in its own bar space. **Weight and brightness carry
the timeframe** — the 4h levels are thickest and brightest, because those are the ones that matter.
Everything on the chart is named the same way: `4h·Z8` is a zone, `5m·S7` a trendline, `1h·FB22` a
gap, `#04` a trade.

In [4]:
fig = lab.figure()
# fig

Open it full width in a browser tab — better than a notebook cell for this.

In [5]:
from quant.viz import open_in_browser
open_in_browser(fig)          # or fig.write_html("fvg.html", include_plotlyjs="cdn")

WindowsPath('C:/Users/Talal Zahid/Desktop/Talal/Binance/notebooks/chart.html')

---
## Appendix · the gap census

`median_bars_to_fill` is the number that decides whether a gap rule can discriminate at all. A
timeframe that produces a gap every few bars has one live on almost every bar, so any "is a gap
present" test on it is a constant, not a filter — which is why `FVG_VETO_TFS` excludes the chart's
own timeframe by default.

In [6]:
lab.census()

,tf,gaps,open,median_size_atr,median_bars_to_fill
0,5m,1128,62,0.78,5.0
1,4h,52,8,0.77,6.0
2,1h,55,6,0.82,4.0
3,15m,108,7,0.80,6.0


Every gap on the chart's own timeframe that is still open, newest last. All three mitigation
thresholds are recorded whatever `FVG_MITIGATION` is set to, so you can see what the other two
would have done without re-running.

In [7]:
_g = lab.gaps[CFG.TF]
_g[_g.dead_i < 0].tail(12)[["id", "side", "born_t", "bot", "top", "size_atr",
                            "touch_i", "ce_i", "full_i"]]

,id,side,born_t,bot,top,size_atr,touch_i,ce_i,full_i
982,FS501,bearish,2026-06-02 04:50:00+00:00,70916.00,70994.47,0.722232,-1,-1,-1
1005,FS513,bearish,2026-06-03 11:35:00+00:00,67192.00,67308.00,1.043317,9660,-1,-1
1008,FS515,bearish,2026-06-03 15:10:00+00:00,66669.86,66760.00,0.508425,9690,-1,-1
1015,FS521,bearish,2026-06-03 22:15:00+00:00,65328.00,65460.99,0.595702,-1,-1,-1
1043,FS535,bearish,2026-06-05 05:40:00+00:00,63216.00,63335.03,0.520919,10191,-1,-1
1072,FB525,bullish,2026-06-07 01:05:00+00:00,60811.99,60982.00,1.324639,-1,-1,-1
1089,FB536,bullish,2026-06-07 20:05:00+00:00,61358.00,61486.00,0.815049,10899,-1,-1
1111,FS562,bearish,2026-06-08 22:05:00+00:00,63581.12,63728.96,1.368409,11221,-1,-1
1118,FS567,bearish,2026-06-09 07:30:00+00:00,63179.98,63300.00,1.054415,11324,-1,-1
1119,FS568,bearish,2026-06-09 08:05:00+00:00,63076.41,63182.95,0.956170,-1,-1,-1
